# 02 — Phenotypic & Complications Extraction
## NIH All of Us — Sickle Cell Disease (SCD) Analysis

| Field | Details |
|-------|---------|
| **Project** | Sickle Cell Disease Multi-Modal Analysis |
| **Dataset** | All of Us Controlled Tier Dataset v8 |
| **Description** | Extracts SCD-specific complications from condition_occurrence, assigns severity categories (None/Mild/Moderate/Severe), performs chi-squared statistical tests by gender, and extracts Hydroxyurea (HU) drug exposure data |
| **Input** | `SCD_Demographics.csv`, All of Us CDR BigQuery (`condition_occurrence`, `drug_exposure`) |
| **Output** | `SCD_Demographics_with_Complications.csv`, `SCD_Complication_Severity_Only.csv`, `SCD_Demographics_with_HU.csv` |

---

## Workflow
1. Define SCD cohort and extract condition occurrences  
2. Map conditions to SCD complications using concept IDs  
3. Assign severity scores (None → Mild → Moderate → Severe)  
4. Analyse complication distribution by gender (chi-squared test)  
5. Extract Hydroxyurea drug exposure per participant  
6. Save all outputs to CSV

---
> ⚠️ **All of Us Compliance:** This notebook must be run inside the All of Us Researcher Workbench. No participant data is stored in this repository.

---
## ⚙️ Before You Begin

> **Prerequisite:** You must have run `01_EHR_demographics_extraction.ipynb` first.  
> This notebook requires `SCD_Demographics.csv` to already exist in your workspace.

### What this notebook does
- Defines the SCD cohort and queries condition occurrences from BigQuery  
- Maps conditions to known SCD complications using OMOP concept IDs  
- Assigns severity scores: **None → Mild → Moderate → Severe**  
- Runs chi-squared statistical tests (gender vs complications)  
- Extracts Hydroxyurea (HU) drug exposure per participant  

### Run order
1. Select **Python 3** kernel  
2. Click **Kernel → Restart & Run All**  
   *OR* run cells one by one with **Shift + Enter**

---

## Step 1 — Define SCD Cohort & Query Condition Occurrences

**What this does:**  
Connects to the All of Us BigQuery CDR and queries the `condition_occurrence` table
to extract all clinical condition records for your 476 SCD participants.

**Key variables:**
- `cdr` — your All of Us workspace CDR path (loaded from environment)
- `scd_cohort_sql` — SQL query filtering to SCD participants only

> ⏱️ *This BigQuery query may take 2–5 minutes.*

In [ ]:
import os
import pandas as pd

# Set CDR environment variable
cdr = os.environ["WORKSPACE_CDR"]

# Step 1: Define SCD Cohort
scd_cohort_sql = f"""
WITH scd_candidates AS (
  SELECT DISTINCT person_id
  FROM `{cdr}.cb_search_all_events`
  WHERE concept_id IN (
    SELECT DISTINCT c.concept_id
    FROM `{cdr}.cb_criteria` c
    JOIN (
      SELECT CAST(id AS STRING) AS id
      FROM `{cdr}.cb_criteria`
      WHERE concept_id IN (24006, 4159748, 22281, 4213628, 315523)
        AND full_text LIKE '%_rank1]%'
    ) a ON (
      c.path LIKE CONCAT('%.', a.id, '.%')
      OR c.path LIKE CONCAT('%.', a.id)
      OR c.path LIKE CONCAT(a.id, '.%')
      OR c.path = a.id
    )
    WHERE is_standard = 1 AND is_selectable = 1
  )
),

filtered_persons AS (
  SELECT cbsp.person_id
  FROM `{cdr}.cb_search_person` cbsp
  WHERE cbsp.person_id IN (
    SELECT person_id FROM `{cdr}.cb_search_person` WHERE age_at_cdr BETWEEN 18 AND 124
  )
  AND cbsp.person_id IN (SELECT person_id FROM scd_candidates)
  AND cbsp.person_id IN (
    SELECT person_id FROM `{cdr}.cb_search_person` WHERE has_whole_genome_variant = 1
  )
  AND cbsp.person_id IN (
    SELECT person_id FROM `{cdr}.person`
    WHERE sex_at_birth_concept_id IN (45878463, 45880669)
      AND race_concept_id IN (8516, 8515, 38003615)
  )
)
SELECT person_id FROM filtered_persons
"""

# Run cohort query
cohort_df = pd.read_gbq(
    scd_cohort_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook"
)

person_ids = tuple(cohort_df['person_id'].tolist())

# Step 2: Mortality with age at death
mortality_sql = f"""
SELECT
  p.person_id,
  p.birth_datetime,
  d.death_date,
  EXTRACT(YEAR FROM d.death_date) - EXTRACT(YEAR FROM p.birth_datetime) AS age_at_death,
  CASE WHEN d.death_date IS NOT NULL THEN 'Deceased' ELSE 'Alive' END AS mortality_status
FROM `{cdr}.person` p
LEFT JOIN `{cdr}.death` d ON p.person_id = d.person_id
WHERE p.person_id IN UNNEST(@person_ids)
"""

# Define parameter
query_params = [{
    "name": "person_ids",
    "parameterType": {"type": "ARRAY", "arrayType": {"type": "INT64"}},
    "parameterValue": {"arrayValues": [{"value": str(pid)} for pid in person_ids]}
}]

# Run query
mortality_df = pd.read_gbq(
    mortality_sql,
    dialect="standard",
    configuration={"query": {"parameterMode": "NAMED", "queryParameters": query_params}}
)

# Display
print("Participants with mortality info:", mortality_df.shape)
mortality_df.head(20)


In [ ]:
import pandas as pd
import os

cdr = os.environ["WORKSPACE_CDR"]

# Define SCD cohort
scd_cohort_sql = f"""
WITH scd_candidates AS (
  SELECT DISTINCT person_id
  FROM `{cdr}.cb_search_all_events`
  WHERE concept_id IN (
    SELECT DISTINCT c.concept_id
    FROM `{cdr}.cb_criteria` c
    JOIN (
      SELECT CAST(id AS STRING) AS id
      FROM `{cdr}.cb_criteria`
      WHERE concept_id IN (24006, 4159748, 22281, 4213628, 315523)
        AND full_text LIKE '%_rank1]%'
    ) a
    ON (c.path LIKE CONCAT('%.', a.id, '.%') OR c.path LIKE CONCAT('%.', a.id) OR c.path LIKE CONCAT(a.id, '.%') OR c.path = a.id)
    WHERE is_standard = 1 AND is_selectable = 1
  )
),
filtered_persons AS (
  SELECT cbsp.person_id
  FROM `{cdr}.cb_search_person` cbsp
  WHERE cbsp.person_id IN (
    SELECT p.person_id
    FROM `{cdr}.cb_search_person` p
    WHERE age_at_cdr BETWEEN 18 AND 124
  )
  AND cbsp.person_id IN (SELECT person_id FROM scd_candidates)
  AND cbsp.person_id IN (
    SELECT person_id FROM `{cdr}.cb_search_person` WHERE has_whole_genome_variant = 1
  )
  AND cbsp.person_id IN (
    SELECT person_id
    FROM `{cdr}.person`
    WHERE sex_at_birth_concept_id IN (45878463, 45880669)
      AND race_concept_id IN (8516, 8515, 38003615)
  )
)
SELECT person_id FROM filtered_persons
"""

# Get cohort
cohort_df = pd.read_gbq(
    scd_cohort_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook"
)

person_ids = tuple(cohort_df['person_id'].tolist())

# ✅ CORRECTED: Mortality query (deceased only)
mortality_sql = f"""
SELECT
  p.person_id,
  DATE(p.birth_datetime) AS birth_date,
  d.death_date,
  DATE_DIFF(DATE(d.death_date), DATE(p.birth_datetime), YEAR) AS age_at_death
FROM `{cdr}.person` p
JOIN `{cdr}.death` d ON p.person_id = d.person_id
WHERE p.person_id IN UNNEST(@person_ids)
"""

query_params = [{
    "name": "person_ids",
    "parameterType": {"type": "ARRAY", "arrayType": {"type": "INT64"}},
    "parameterValue": {"arrayValues": [{"value": str(pid)} for pid in person_ids]}
}]

# Run mortality query
mortality_df = pd.read_gbq(
    mortality_sql,
    dialect="standard",
    configuration={"query": {"parameterMode": "NAMED", "queryParameters": query_params}}
)

# Show deceased with age
print("✅ Deceased participants:", mortality_df.shape[0])
mortality_df.head(10)


---
## Step 2 — Handle Mortality Data

**What this does:**  
Saves mortality (deceased participant) data and prepares person IDs for downstream queries.

> ℹ️ Deceased participants were already excluded in Notebook 01 via the cohort filter,
> but this step captures any mortality events recorded during the study period.

In [ ]:
# Save to CSV in current notebook environment
mortality_df.to_csv("deceased_participants.csv", index=False)


In [ ]:
# Convert person_ids to a comma-separated string
id_string = ",".join(str(pid) for pid in person_ids)

# Rebuild mortality SQL with embedded IDs
mortality_sql = f"""
SELECT
  p.person_id,
  DATE(p.birth_datetime) AS birth_date,
  d.death_date,
  DATE_DIFF(DATE(d.death_date), DATE(p.birth_datetime), YEAR) AS age_at_death
FROM `{cdr}.person` p
JOIN `{cdr}.death` d ON p.person_id = d.person_id
WHERE p.person_id IN ({id_string})
"""

# Run corrected query
mortality_df = pd.read_gbq(
    mortality_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ)
)


---
## Step 3 — Load & Inspect Merged Demographics + Complications

**What this does:**  
Loads `SCD_Demographics.csv` and the initial complications data, then inspects
column structure to confirm successful merge.

**Expected output:** Column names printed for both dataframes.

In [ ]:
import pandas as pd

# Load your datasets
demo = pd.read_csv("SCD_Demographics.csv")
comp = pd.read_csv("scd_complication_counts_all.csv")

# Inspect to find the common column
print("Demographics columns:", demo.columns.tolist())
print("Complications columns:", comp.columns.tolist())


In [ ]:
import pandas as pd

demo = pd.read_csv("SCD_Demographics.csv")
comp = pd.read_csv("scd_complication_counts_all.csv")

# Since complications is not per patient, just add summary counts as separate columns
n_patients = comp.set_index("Complication")["n_patients"].to_dict()
pct_patients = comp.set_index("Complication")["pct_of_476"].to_dict()

# Example: add overall complication summaries to demographics metadata
summary_info = pd.DataFrame({
    "total_participants": [len(demo)],
    **{f"{k}_n": [v] for k,v in n_patients.items()},
    **{f"{k}_pct": [v] for k,v in pct_patients.items()}
})

# Save separately
summary_info.to_csv("SCD_Demographics_with_Complications_summary.csv", index=False)


---
## Step 4 — Calculate Participant Ages

**What this does:**  
Computes each participant's age from their `date_of_birth` field and
plots the age distribution across the SCD cohort.

**Expected output:** Age distribution histogram with a normal distribution overlay.

In [ ]:
import numpy as np

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm
from datetime import datetime, timezone

# Load your data (make sure df is defined)

df = pd.read_csv ("SCD_Demographics.csv")

# 1. Convert date_of_birth to datetime
df['date_of_birth'] = pd.to_datetime(df['date_of_birth'], errors='coerce')

# 2. Drop rows where date conversion failed (optional but recommended)
df = df.dropna(subset=['date_of_birth'])

# 3. Calculate age
now = datetime.now(timezone.utc)
df['age'] = df['date_of_birth'].apply(lambda dob: (now - dob).days / 365.25)

# 4. Plot histogram
plt.figure(figsize=(10, 6))
n, bins, patches = plt.hist(df['age'], bins=20, density=True, alpha=0.6, edgecolor='black', label='Age distribution')

# 5. Fit normal distribution
mu, std = norm.fit(df['age'])

# 6. Plot normal curve
xmin, xmax = plt.xlim()
x = np.linspace(xmin, xmax, 100)
p = norm.pdf(x, mu, std)
plt.plot(x, p, 'k', linewidth=2, label=f'Normal fit\nμ={mu:.2f}, σ={std:.2f}')

# 7. Final plot settings
plt.title('Age Distribution with Normal Curve Fit')
plt.xlabel('Age (years)')
plt.ylabel('Density')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
# Save to CSV with person_id and age
df[['person_id', 'age']].to_csv("participant_ages.csv", index=False)


---
## Step 5 — Query Phenotype / Condition Data from BigQuery

**What this does:**  
Runs a SQL query to pull all condition records with concept names,
then inspects the resulting dataframe structure.

In [ ]:
phenotype_sql = f"""
SELECT
    c.person_id,
    c.condition_concept_id,
    cond.concept_name as condition_name,
    c.condition_start_date,
    c.condition_type_concept_id
FROM
    `{workspace}.condition_occurrence` c
LEFT JOIN
    `{workspace}.concept` cond ON c.condition_concept_id = cond.concept_id
WHERE
    c.person_id IN (
        SELECT DISTINCT person_id FROM `{workspace}.person`
        WHERE person_id IN (
            SELECT person_id FROM `{workspace}.cb_search_person`
            WHERE has_whole_genome_variant = 1
            AND person_id IN (
                SELECT person_id FROM `{workspace}.person`
                WHERE race_concept_id IN (8516, 8515, 38003615)
                AND sex_at_birth_concept_id IN (45878463, 45880669)
                AND EXISTS (
                    SELECT 1 FROM `{workspace}.cb_search_person` cbp
                    WHERE cbp.person_id = person_id
                    AND age_at_cdr BETWEEN 18 AND 124
                )
            )
        )
    )
    AND c.condition_concept_id IN (
        -- Add SCD and related condition concept IDs
        190081,  -- Sickle cell anemia
        443832,  -- Sickle cell trait
        432794,  -- Hemoglobinopathy
        201826,  -- Anemia
        43528895 -- Add other relevant IDs here
    )
ORDER BY c.person_id, c.condition_start_date
"""

phenotype_df = pd.read_gbq(
    phenotype_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook"
)

In [ ]:
import pandas as pd

cond = pd.read_csv("SCD_Demographics.csv")
print(cond.columns)


In [ ]:
import pandas as pd
from pathlib import Path

# ---------- INPUTS ----------
DEMOS_PATH = "SCD_Demographics.csv"                 # your 476 participants
COND_PATH  = "your_condition_occurrence_data.csv"   # long table: person_id + condition_concept_id (+ dates optional)
COMP_PATH  = "scd_complications_concept_ids.csv"    # your (Complication, concept_id) map (or comment this and use inline list)

# ---------- LOAD DEMOGRAPHICS (476) ----------
demos = pd.read_csv(DEMOS_PATH, dtype={"person_id": int})
study_person_ids = set(demos["person_id"])

# ---------- LOAD CONDITIONS (LONG) ----------
cond = pd.read_csv(COND_PATH)

# standardize columns to lowercase for safer lookup
cond.columns = [c.lower() for c in cond.columns]

# detect columns
if "person_id" not in cond.columns:
    raise KeyError("Could not find 'person_id' in condition file columns: " + ", ".join(cond.columns))
if "condition_concept_id" not in cond.columns:
    # try a couple of common variants; add more if needed
    candidates = [c for c in cond.columns if c.replace("_", "") == "conditionconceptid"]
    if candidates:
        cond.rename(columns={candidates[0]: "condition_concept_id"}, inplace=True)
    else:
        raise KeyError("Could not find 'condition_concept_id' in condition file columns: " + ", ".join(cond.columns))

# keep only the 476 participants
cond = cond[cond["person_id"].isin(study_person_ids)]

# ---------- LOAD COMPLICATION LIST ----------
if Path(COMP_PATH).exists():
    scd_df = pd.read_csv(COMP_PATH, dtype={"concept_id": int})
    # normalize column names if needed
    scd_df.columns = [c.strip() for c in scd_df.columns]
    if "Complication" not in scd_df.columns or "concept_id" not in scd_df.columns:
        raise KeyError("Your complications file must have columns: 'Complication', 'concept_id'")
else:
    # Inline fallback if the CSV isn't available
    scd_complications = [
        ("Haemolytic anaemia/Chronic haemolytic anaemia", 435503),
        ("Aplastic crisis / Aplastic Anemia", 137829),
        ("Iron overload", 4246084),
        ("Vaso-occlusive crises", 35626024),
        ("Dactylitis", 4131942),
        ("Hand and foot syndrome", 4159748),
        ("Chronic pain", 436096),
        ("Avascular necrosis/AVN", 4287786),
        ("Leg ulcers", 197304),
        ("Ischaemic stroke", 4310996),
        ("Transient ischemic attacks (TIAs)", 4043734),
        ("Cognitive impairment", 40480615),
        ("Seizures", 4106574),
        ("Acute chest syndrome", 254062),
        ("Pulmonary hypertension", 4322024),
        ("Restrictive lung disease", 4266931),
        ("Obstructive lung disease", 255573),
        ("Sleep-disordered breathing", 44783631),
        ("Cardiomyopathy", 321319),
        ("Haematuria", 79864),
        ("Proteinuria", 75650),
        ("Albuminuria", 4168705),
        ("Chronic kidney disease (CKD)", 46271022),
        ("End-stage kidney disease (ESKD)", 193782),
        ("Focal segmental glomerulosclerosis (FSGS)", 4030513),
        ("Renal papillary necrosis", 4152839),
        ("Nocturia", 40304526),
        ("Enuresis", 193874),
        ("Cholestasis", 4143915),
        ("Cholelithiasis", 444367),
        ("Gallstones", 196456),
        ("Liver cirrhosis", 4064161),
        ("Osteomyelitis", 141663),
        ("Growth delay", 4031877),
        ("Sepsis", 132797),
        ("Meningitis", 435785),
        ("Parvovirus B19 infection", 134569),
        ("Depression", 440383),
        ("Anxiety", 441542),
        ("Cognitive delay", 4137543),
        ("Social isolation", 4309238),
        ("Chronic stress", 4138454),
        ("Delayed puberty", 4266651),
        ("Infertility", 201909),
        ("Infertility", 198197),  # duplicate name on purpose; code below sums them
        ("Priapism", 315586),
        ("Miscarriage", 4067106),
        ("Preeclampsia", 439393),
        ("Renal transplant", 4324887),
        ("Splenectomy", 4120626),
        ("Bone marrow transplant", 42537745),
        ("Cerebral aneurysm", 4029497),
        ("Sickle retinopathy", 4021365),
        ("Central retinal artery occlusion", 437540),
        ("Retinal detachment", 378414),
        ("Vitreous haemorrhage", 315276),
    ]
    scd_df = pd.DataFrame(scd_complications, columns=["Complication", "concept_id"])

# ---------- FILTER TO COMPLICATIONS ----------
cond_filt = cond[cond["condition_concept_id"].isin(scd_df["concept_id"])]

# attach names
cond_named = cond_filt.merge(
    scd_df,
    left_on="condition_concept_id",
    right_on="concept_id",
    how="left",
)

# ---------- COUNT UNIQUE PATIENTS PER COMPLICATION ----------
counts = (cond_named
          .groupby(["Complication"], dropna=False)
          .agg(n_patients=("person_id", "nunique"))
          .reset_index())

# add percentage of the 476
denom = len(study_person_ids) if len(study_person_ids) > 0 else 1
counts["pct_of_476"] = (counts["n_patients"] / denom * 100).round(1)

# sort and keep top 10
top10 = counts.sort_values("n_patients", ascending=False).head(10)

# nice printout
pd.set_option("display.max_rows", None)
print("\nTop 10 complications by unique patients (n = %d):" % denom)
print(top10.to_string(index=False))

# ---------- OPTIONAL DIAGNOSTICS ----------
# Which of your listed complication concept_ids never appear?
present_ids = set(cond_filt["condition_concept_id"].unique())
missing_ids = set(scd_df["concept_id"]) - present_ids
if missing_ids:
    miss = (scd_df[scd_df["concept_id"].isin(missing_ids)]
            .sort_values("Complication")[["Complication","concept_id"]])
    print("\nThese complication IDs were NOT observed in the 476 participants:")
    print(miss.to_string(index=False))


---
## Step 6 — Map Conditions to SCD Complications

**What this does:**  
Merges complication counts with a full reference list of SCD complications
so that participants with zero occurrences of a complication still appear
in the output (with a count of 0 rather than being missing).

**Expected output:** Full complication table with `n_patients` column for all complications.

In [ ]:
# Merge counts with full complication list so missing ones get 0
counts_full = scd_df.merge(counts, on="Complication", how="left").fillna({"n_patients": 0})
counts_full["n_patients"] = counts_full["n_patients"].astype(int)

# Add percentage column
denom = len(study_person_ids) if len(study_person_ids) > 0 else 1
counts_full["pct_of_476"] = (counts_full["n_patients"] / denom * 100).round(1)

# Sort by count desc, then alphabetically, and take top 10
top10 = counts_full.sort_values(["n_patients", "Complication"], ascending=[False, True]).head(10)

print("\nTop 10 complications by unique patients (n = %d):" % denom)
print(top10.to_string(index=False))


In [ ]:
import pandas as pd

# -------- INPUT FILES --------
DEMOS_PATH = "SCD_Demographics.csv"               # your 476 participants
COND_PATH  = "your_condition_occurrence_data.csv" # condition_occurrence-style data

# -------- LOAD DEMOGRAPHICS --------
demos = pd.read_csv(DEMOS_PATH, dtype={"person_id": int})
study_ids = set(demos["person_id"])

# -------- LOAD CONDITIONS --------
cond = pd.read_csv(COND_PATH)
cond.columns = [c.lower() for c in cond.columns]
if "person_id" not in cond.columns or "condition_concept_id" not in cond.columns:
    raise KeyError(f"Need columns person_id and condition_concept_id in {COND_PATH}")
cond = cond[cond["person_id"].isin(study_ids)]

# -------- CREATE COMPLICATION LIST --------
scd_complications = [
    ("Haemolytic anaemia/Chronic haemolytic anaemia", 435503),
    ("Aplastic crisis / Aplastic Anemia", 137829),
    ("Iron overload", 4246084),
    ("Vaso-occlusive crises", 35626024),
    ("Dactylitis", 4131942),
    ("Hand and foot syndrome", 4159748),
    ("Chronic pain", 436096),
    ("Avascular necrosis/AVN", 4287786),
    ("Leg ulcers", 197304),
    ("Ischaemic stroke", 4310996),
    ("Transient ischemic attacks (TIAs)", 4043734),
    ("Cognitive impairment", 40480615),
    ("Seizures", 4106574),
    ("Acute chest syndrome", 254062),
    ("Pulmonary hypertension", 4322024),
    ("Restrictive lung disease", 4266931),
    ("Obstructive lung disease", 255573),
    ("Sleep-disordered breathing", 44783631),
    ("Cardiomyopathy", 321319),
    ("Haematuria", 79864),
    ("Proteinuria", 75650),
    ("Albuminuria", 4168705),
    ("Chronic kidney disease (CKD)", 46271022),
    ("End-stage kidney disease (ESKD)", 193782),
    ("Focal segmental glomerulosclerosis (FSGS)", 4030513),
    ("Renal papillary necrosis", 4152839),
    ("Nocturia", 40304526),
    ("Enuresis", 193874),
    ("Cholestasis", 4143915),
    ("Cholelithiasis", 444367),
    ("Gallstones", 196456),
    ("Liver cirrhosis", 4064161),
    ("Osteomyelitis", 141663),
    ("Growth delay", 4031877),
    ("Sepsis", 132797),
    ("Meningitis", 435785),
    ("Parvovirus B19 infection", 134569),
    ("Depression", 440383),
    ("Anxiety", 441542),
    ("Cognitive delay", 4137543),
    ("Social isolation", 4309238),
    ("Chronic stress", 4138454),
    ("Delayed puberty", 4266651),
    ("Infertility", 201909),
    ("Infertility", 198197),
    ("Priapism", 315586),
    ("Miscarriage", 4067106),
    ("Preeclampsia", 439393),
    ("Renal transplant", 4324887),
    ("Splenectomy", 4120626),
    ("Bone marrow transplant", 42537745),
    ("Cerebral aneurysm", 4029497),
    ("Sickle retinopathy", 4021365),
    ("Central retinal artery occlusion", 437540),
    ("Retinal detachment", 378414),
    ("Vitreous haemorrhage", 315276)
]
scd_df = pd.DataFrame(scd_complications, columns=["Complication", "concept_id"])

# -------- COUNT --------
cond_filt = cond[cond["condition_concept_id"].isin(scd_df["concept_id"])]
cond_named = cond_filt.merge(scd_df, left_on="condition_concept_id", right_on="concept_id", how="left")
counts = (cond_named.groupby("Complication", dropna=False)
          .agg(n_patients=("person_id", "nunique"))
          .reset_index())

# Merge to include zeros
all_counts = scd_df.groupby("Complication", as_index=False) \
    .agg(concept_ids=("concept_id", lambda s: sorted(set(s)))) \
    .merge(counts, on="Complication", how="left") \
    .fillna({"n_patients": 0})
all_counts["n_patients"] = all_counts["n_patients"].astype(int)
denom = len(study_ids) or 1
all_counts["pct_of_476"] = (all_counts["n_patients"] / denom * 100).round(1)
all_counts["concept_ids"] = all_counts["concept_ids"].apply(lambda ids: ";".join(map(str, ids)))
all_counts = all_counts.sort_values(["n_patients", "Complication"], ascending=[False, True])

# -------- TOP 10 --------
top10 = all_counts.head(10)

# -------- SAVE --------
all_counts.to_csv("scd_complication_counts_all.csv", index=False)
top10.to_csv("scd_complication_top10.csv", index=False)

print(" Saved:")
print(" - scd_complication_counts_all.csv")
print(" - scd_complication_top10.csv")


---
## Step 7 — Define SCD Complication Concept IDs

**What this does:**  
Creates the reference mapping of SCD complication names to their
OMOP concept IDs. This is the key lookup table used throughout the pipeline.

**Complications covered:**
- Haemolytic anaemia, Aplastic crisis, VOC, Dactylitis, Acute chest syndrome
- Restrictive lung disease, Stroke, Osteomyelitis, Preeclampsia, Liver cirrhosis
- And more (full list in the cell below)

In [ ]:
# import pandas as pd

# # Define a list of complications and associated concept IDs
# scd_complications = [
#     ("Haemolytic anaemia/Chronic haemolytic anaemia", 435503),
#     ("Aplastic crisis / Aplastic Anemia", 137829),
#     ("Iron overload", 4246084),
#     ("Vaso-occlusive crises", 35626024),
#     ("Dactylitis", 4131942),
#     ("hand and foot syndrome Anemia", 4159748),
#     ("Chronic pain", 436096),
#     ("Avascular necrosis/AVN", 4287786),
#     ("Leg ulcers / ULCER OF LOWER EXTREMITY", 197304),
#     ("Ischaemic stroke", 4310996),
#     ("Transient ischemic attacks (TIAs)", 4043734),
#     ("Cognitive impairment / disorder", 40480615),
#     ("Seizures", 4106574),
#     ("Acute chest syndrome", 254062),
#     ("Pulmonary hypertension", 4322024),
#     ("Restrictive lung disease", 4266931),
#     ("Obstructive lung disease", 255573),
#     ("Sleep-disordered breathing", 44783631),
#     ("Cardiomyopathy", 321319),
#     ("Haematuria", 79864),
#     ("Proteinuria", 75650),
#     ("Albuminuria", 4168705),
#     ("Chronic kidney disease (CKD)", 46271022),
#     ("End-stage kidney disease (ESKD)", 193782),
#     ("Focal segmental glomerulosclerosis (FSGS)", 4030513),
#     ("Renal papillary necrosis", 4152839),
#     ("Nocturia", 40304526),
#     ("Enuresis", 193874),
#     ("Cholestasis", 4143915),
#     ("Cholelithiasis", 444367),
#     ("gallstones", 196456),
#     ("Liver cirrhosis", 4064161),
#     ("Osteomyelitis", 141663),
#     ("Growth delay", 4031877),
#     ("Sepsis", 132797),
#     ("Meningitis", 435785),
#     ("Parvovirus B19 infection", 134569),
#     ("Depression", 440383),
#     ("Anxiety", 441542),
#     ("Cognitive delay", 4137543),
#     ("Social isolation", 4309238),
#     ("Chronic stress", 4138454),
#     ("Delayed puberty", 4266651),
#     ("Infertility (Female / Male)", 201909),
#     ("Infertility (Female / Male)", 198197),
#     ("Priapism", 315586),
#     ("Miscarriage", 4067106),
#     ("Preeclampsia", 439393),
#     ("Renal transplant", 4324887),
#     ("Splenectomy", 4120626),
#     ("Bone marrow transplant", 42537745),
#     ("Cerebral aneurysm", 4029497),
#     ("Sickle retinopathy", 4021365),
#     ("Central retinal artery occlusion", 437540),
#     ("Retinal detachment", 378414),
#     ("Vitreous haemorrhage", 315276)
# ]

# # Create DataFrame
# scd_df = pd.DataFrame(scd_complications, columns=["Complication", "concept_id"])

# # Show the top 5 rows
# print(scd_df.head(15))


# # Save to CSV
# scd_df.to_csv("scd_complications_concept_ids.csv", index=False)


In [ ]:
import pandas as pd

# List of complications with their concept IDs
scd_complications = [
    ("Haemolytic anaemia/Chronic haemolytic anaemia", 435503),
    ("Aplastic crisis / Aplastic Anemia", 137829),
    ("Iron overload", 4246084),
    ("Vaso-occlusive crises", 35626024),
    ("Dactylitis", 4131942),
    ("hand and foot syndrome Anemia", 4159748),
    ("Chronic pain", 436096),
    ("Avascular necrosis/AVN", 4287786),
    ("Leg ulcers / ULCER OF LOWER EXTREMITY", 197304),
    ("Ischaemic stroke", 4310996),
    ("Transient ischemic attacks (TIAs)", 4043734),
    ("Cognitive impairment / disorder", 40480615),
    ("Seizures", 4106574),
    ("Acute chest syndrome", 254062),
    ("Pulmonary hypertension", 4322024),
    ("Restrictive lung disease", 4266931),
    ("Obstructive lung disease", 255573),
    ("Sleep-disordered breathing", 44783631),
    ("Cardiomyopathy", 321319),
    ("Haematuria", 79864),
    ("Proteinuria", 75650),
    ("Albuminuria", 4168705),
    ("Chronic kidney disease (CKD)", 46271022),
    ("End-stage kidney disease (ESKD)", 193782),
    ("Focal segmental glomerulosclerosis (FSGS)", 4030513),
    ("Renal papillary necrosis", 4152839),
    ("Nocturia", 40304526),
    ("Enuresis", 193874),
    ("Cholestasis", 4143915),
    ("Cholelithiasis", 444367),
    ("gallstones", 196456),
    ("Liver cirrhosis", 4064161),
    ("Osteomyelitis", 141663),
    ("Growth delay", 4031877),
    ("Sepsis", 132797),
    ("Meningitis", 435785),
    ("Parvovirus B19 infection", 134569),
    ("Depression", 440383),
    ("Anxiety", 441542),
    ("Cognitive delay", 4137543),
    ("Social isolation", 4309238),
    ("Chronic stress", 4138454),
    ("Delayed puberty", 4266651),
    ("Infertility (Female / Male)", 201909),
    ("Infertility (Female / Male)", 198197),
    ("Priapism", 315586),
    ("Miscarriage", 4067106),
    ("Preeclampsia", 439393),
    ("Renal transplant", 4324887),
    ("Splenectomy", 4120626),
    ("Bone marrow transplant", 42537745),
    ("Cerebral aneurysm", 4029497),
    ("Sickle retinopathy", 4021365),
    ("Central retinal artery occlusion", 437540),
    ("Retinal detachment", 378414),
    ("Vitreous haemorrhage", 315276)
]

# Create DataFrame
scd_df = pd.DataFrame(scd_complications, columns=["Complication", "concept_id"])

# Save to CSV
scd_df.to_csv("scd_complications_concept_ids.csv", index=False)

print("Saved file: scd_complications_concept_ids.csv with", scd_df.shape[0], "rows")


---
## Step 8 — Flag Complications Per Participant

**What this does:**  
Merges the complication concept IDs with condition occurrence data to create
a binary flag (0/1) for each complication per participant.

**Expected output:** `demo_with_complications` dataframe — one row per participant,
one column per complication.

In [ ]:
import pandas as pd

# Load your files
demo = pd.read_csv("SCD_Demographics.csv")   # has person_id
comp_list = pd.read_csv("scd_complications_concept_ids.csv")  # Complication, concept_id
events = pd.read_csv("your_condition_occurrence_data.csv")    # person_id, condition_concept_id

# Make sure IDs are integers
comp_list["concept_id"] = comp_list["concept_id"].astype(int)
events["condition_concept_id"] = events["condition_concept_id"].astype(int)

# Merge events with complication dictionary (adds Complication name)
events_named = events.merge(comp_list, left_on="condition_concept_id", right_on="concept_id", how="inner")

# Pivot long → wide (each complication becomes a column, values = counts)
comp_matrix = pd.crosstab(events_named["person_id"], events_named["Complication"]).reset_index()

# Optional: convert counts to binary indicators (0/1)
comp_matrix_bin = comp_matrix.copy()
for c in comp_matrix_bin.columns:
    if c != "person_id":
        comp_matrix_bin[c] = (comp_matrix_bin[c] > 0).astype(int)

# Merge demographics with complications
merged = demo.merge(comp_matrix_bin, on="person_id", how="left")

# Fill NaNs (people with no complications → 0)
for c in merged.columns:
    if c not in demo.columns:
        merged[c] = merged[c].fillna(0).astype(int)

# Save the final dataset
merged.to_csv("SCD_Demographics_with_Complications.csv", index=False)

print(" Created SCD_Demographics_with_Complications.csv with shape:", merged.shape)


In [ ]:
demo_with_complications.gender.value_counts()

---
## Step 9 — Visualise Complication Distribution

**What this does:**  
Creates bar charts showing how many participants have each complication,
broken down by gender where applicable.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Define comorbidity columns
comorbidities = [
    'Aplastic_crisis', 'Hand_foot_syndrome', 'Liver_cirrhosis',
    'Osteomyelitis', 'Preeclampsia', 'Pulmonary_hypertension',
    'Restrictive_lung_disease', 'Sickle_retinopathy',
    'Vitreous_haemorrhage', 'Dactylitis'
]

# --- 1. Comorbidity burden per patient ---
demo_with_complications['comorbidity_count'] = demo_with_complications[comorbidities].sum(axis=1)

plt.figure(figsize=(10, 6))
sns.histplot(demo_with_complications['comorbidity_count'], bins=range(0, demo_with_complications['comorbidity_count'].max()+2), kde=False)
plt.title('Distribution of Comorbidity Count Per Patient')
plt.xlabel('Number of Comorbidities')
plt.ylabel('Number of Patients')
plt.grid(True)
plt.tight_layout()
plt.show()

# --- 2. Proportion of patients with each comorbidity ---
comorbidity_prevalence = demo_with_complications[comorbidities].mean().sort_values(ascending=False) * 100

plt.figure(figsize=(12, 6))
sns.barplot(x=comorbidity_prevalence.index, y=comorbidity_prevalence.values)
plt.title('Comorbidity Prevalence in the SCD Cohort')
plt.ylabel('Percentage of Patients (%)')
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y')
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Define complication columns
complication_cols = [
    'Dactylitis', 'Restrictive_lung_disease', 'Preeclampsia',
    'Sickle_retinopathy', 'Osteomyelitis', 'Aplastic_crisis',
    'Liver_cirrhosis', 'Vitreous_haemorrhage',
    'Hand_foot_syndrome', 'Pulmonary_hypertension'
]

# Clean up names for plotting
complication_labels = {
    'Dactylitis': 'Dactylitis',
    'Restrictive_lung_disease': 'Restrictive lung disease',
    'Preeclampsia': 'Preeclampsia',
    'Sickle_retinopathy': 'Sickle retinopathy',
    'Osteomyelitis': 'Osteomyelitis',
    'Aplastic_crisis': 'Aplastic crisis / Aplastic Anemia',
    'Liver_cirrhosis': 'Liver cirrhosis',
    'Vitreous_haemorrhage': 'Vitreous haemorrhage',
    'Hand_foot_syndrome': 'hand and foot syndrome Anemia',
    'Pulmonary_hypertension': 'Pulmonary hypertension'
}

# --- 1. Horizontal bar plot with counts and percentages ---
total_patients = len(demo_with_complications)
complication_counts = demo_with_complications[complication_cols].sum().sort_values(ascending=False)
complication_percent = (complication_counts / total_patients * 100).round(1)

# Plot
plt.figure(figsize=(10, 6))
sns.barplot(x=complication_counts.values, y=[complication_labels[c] for c in complication_counts.index])
plt.title('Top 10 SCD Complications')
plt.xlabel('Number of Cases')
plt.tight_layout()
plt.show()

# Optional: print with percentage
for comp, count in complication_counts.items():
    print(f"{complication_labels[comp]}: {count} ({complication_percent[comp]}%)")

import matplotlib.pyplot as plt

# Optional: print with percentage
for comp, count in complication_counts.items():
    print(f"{complication_labels[comp]}: {count} ({complication_percent[comp]}%)")

# --- 2. Sex comparison (only Male and Female) ---
df_mf = demo_with_complications[demo_with_complications['sex_at_birth'].isin(['Male', 'Female'])]

sex_comparison = df_mf.groupby('sex_at_birth')[complication_cols].mean().T * 100
sex_comparison = sex_comparison.rename(index=complication_labels).round(1)

# Plot sex comparison
plt.figure(figsize=(12, 6))
sex_comparison[['Male', 'Female']].plot(kind='bar', figsize=(12, 6), width=0.8)
plt.title('Comorbidity Prevalence by Sex at Birth (Female vs Male Only)')
plt.ylabel('Percentage of Patients with Complication')
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y')
plt.tight_layout()
plt.legend(title='Sex at Birth')
plt.show()



---
## Step 10 — Chi-Squared Test: Gender vs Complications

**What this does:**  
Runs a chi-squared statistical test for each complication to determine
whether there is a significant difference in complication rates between
Male and Female participants.

**Output:** p-values for each complication. Values < 0.05 indicate
a statistically significant gender difference.

In [ ]:
print(demo_with_complications.columns.tolist())


In [ ]:
from scipy.stats import chi2_contingency
import pandas as pd

# Filter to Male and Female only
df_mf = demo_with_complications[demo_with_complications['gender'].isin(['Male', 'Female'])]

results = []

print("Chi-square test results for gender vs. comorbidities:\n")

for col in complication_cols:
    # Build contingency table (gender vs presence of condition)
    contingency = pd.crosstab(df_mf['gender'], df_mf[col])

    # Ensure both 0 and 1 columns exist
    for val in [0, 1]:
        if val not in contingency.columns:
            contingency[val] = 0
    contingency = contingency[[0, 1]]  # consistent order

    # Only proceed if expected table won't contain zeros
    if (contingency.sum(axis=0) > 0).all() and (contingency.sum(axis=1) > 0).all():
        try:
            chi2, p, dof, expected = chi2_contingency(contingency)
            results.append({
                'Complication': complication_labels[col],
                'Chi2': round(chi2, 3),
                'p-value': round(p, 4)
            })
        except ValueError:
            # Skip this complication if test fails
            continue

# Show results
results_df = pd.DataFrame(results).sort_values(by='p-value')
results_df

---
## Step 11 — Complication Prevalence Visualisations

**What this does:**  
Plots complication prevalence charts from the reference complication list
and the actual participant data, including top-10 complication rankings.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# STEP 1: Load corrected SCD complications from CSV
scd_df = pd.read_csv("scd_complications_concept_ids.csv")

# STEP 2: (Optional) Simulate complication counts
# Replace this section with real counts from BigQuery if available
# For demo purposes only:
import numpy as np
np.random.seed(42)
scd_df["case_count"] = np.random.randint(500, 5000, size=len(scd_df))

# STEP 3: Sort by case count and get Top 10
top_10_df = scd_df.sort_values(by="case_count", ascending=False).head(10)

# STEP 4: Save full and top 10 DataFrames as CSVs
scd_df.to_csv("all_scd_complications_with_counts.csv", index=False)
top_10_df.to_csv("top_10_scd_complications.csv", index=False)

# STEP 5: Plot Top 10 Complications
plt.figure(figsize=(10, 6))
bars = plt.barh(top_10_df["Complication"], top_10_df["case_count"])
plt.xlabel("Case Count")
plt.title("Top 10 SCD Complications")
plt.gca().invert_yaxis()
plt.grid(axis="x", linestyle="--", alpha=0.7)
plt.tight_layout()

# Save the plot
plt.savefig("top_10_scd_complications_plot.png", dpi=300)
plt.show()

# STEP 6: Print Top 10 Table (for thesis inclusion)
print("Top 10 SCD Complications:")
print(top_10_df[["Complication", "concept_id", "case_count"]])


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Step 1: Replace this with your real data from BigQuery
# Patient counts (not event counts), one per complication
df = pd.DataFrame({
    "Complication": [
        "Dactylitis", "Restrictive lung disease", "Preeclampsia",
        "Sickle retinopathy", "Osteomyelitis", "Aplastic crisis",
        "Liver cirrhosis", "Vitreous haemorrhage", 
        "Hand and foot syndrome", "Pulmonary hypertension"
    ],
    "concept_id": [
        4131942, 4266931, 439393, 4021365, 141663,
        137829, 4064161, 315276, 4159748, 4322024
    ],
    "patient_count": [120, 95, 90, 85, 80, 78, 74, 71, 70, 69]  # Replace with your real values
})

# Step 2: Optional: Add percentage
df["% of SCD Patients"] = (df["patient_count"] / 476 * 100).round(1)

# Step 3: Plot bar chart
plt.style.use("seaborn-v0_8-whitegrid")
plt.figure(figsize=(10, 6))
bars = plt.barh(df["Complication"], df["patient_count"], color="steelblue")


plt.xlabel("Number of Unique Patients")
plt.title("Top 10 SCD Complications (n = 476)")
plt.gca().invert_yaxis()
plt.tight_layout()

# Step 5: Save plot and data
plt.savefig("top_10_scd_complications_single.png", dpi=300)
df.to_csv("top_10_scd_complications_single.csv", index=False)
plt.show()

# Optional: print for thesis table
print(df[["Complication", "concept_id", "patient_count"]])


In [ ]:
import pandas as pd

# Load your complications file
comp_df = pd.read_csv("scd_complication_counts_all.csv")

# Preview columns
print(comp_df.columns)


In [ ]:
import pandas as pd

# Load the complications file
df = pd.read_csv("scd_complication_counts_all.csv")

# Filter rows where concept_id is VOC (35626024)
voc_id = 35626024
voc_df = df[df["concept_ids"] == voc_id]

# Show results
print(f"✅ VOC entries found: {voc_df.shape[0]}")
print(voc_df.head(10))


---
## Step 12 — Build Full Complication Matrix for All 476 Participants

**What this does:**  
Extracts condition occurrences directly from BigQuery for all 476 participants
and creates a complete binary complication matrix (participant × complication).

> ⏱️ *This step involves a BigQuery query and may take 2–4 minutes.*

In [ ]:
import pandas as pd
import os

# Load your participant list from your SCD_Demographics.csv
df_demo = pd.read_csv("SCD_Demographics.csv")
person_ids = df_demo["person_id"].unique().tolist()

# Define your top 10 complication concept IDs
complication_ids = [
    4131942,  # Dactylitis
    4266931,  # Restrictive lung disease
    439393,   # Preeclampsia
    4021365,  # Sickle retinopathy
    141663,   # Osteomyelitis
    137829,   # Aplastic crisis
    4064161,  # Liver cirrhosis
    315276,   # Vitreous haemorrhage
    4159748,  # Hand and foot syndrome
    4322024   # Pulmonary hypertension
]

# Format for SQL IN clause
person_id_list_str = ", ".join([str(pid) for pid in person_ids])
concept_id_list_str = ", ".join([str(cid) for cid in complication_ids])

# Construct SQL query
condition_sql = f"""
SELECT person_id, condition_concept_id
FROM `{os.environ["WORKSPACE_CDR"]}.condition_occurrence`
WHERE person_id IN ({person_id_list_str})
AND condition_concept_id IN ({concept_id_list_str})
"""

# Run the query with pandas.read_gbq
df_conditions = pd.read_gbq(
    condition_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook"
)

# Preview and save
print(df_conditions.head(476))
df_conditions.to_csv("your_condition_occurrence_data.csv", index=False)


In [ ]:
import pandas as pd

# Load your condition_occurrence file
cond = pd.read_csv("your_condition_occurrence_data.csv", dtype={"person_id": int, "condition_concept_id": int})

# Your complication IDs
complication_ids = [
    4131942,  # Dactylitis
    4266931,  # Restrictive lung disease
    439393,   # Preeclampsia
    4021365,  # Sickle retinopathy
    141663,   # Osteomyelitis
    137829,   # Aplastic crisis
    4064161,  # Liver cirrhosis
    315276,   # Vitreous haemorrhage
    4159748,  # Hand and foot syndrome
    4322024   # Pulmonary hypertension
]

# Find which IDs from your list are present in the file
present_ids = set(cond["condition_concept_id"]).intersection(complication_ids)
missing_ids = set(complication_ids) - present_ids

print("Present IDs:", sorted(present_ids))
if missing_ids:
    print("Missing IDs:", sorted(missing_ids))
else:
    print("All listed complication IDs are present in the file.")

# (Optional) Find any IDs in file that are NOT in your list
extra_ids = set(cond["condition_concept_id"]) - set(complication_ids)
if extra_ids:
    print(" Extra condition_concept_ids found in file (not in your list):", sorted(extra_ids))


In [ ]:
import pandas as pd

# Load your complications reference list
scd_df = pd.read_csv("scd_complications_concept_ids.csv", dtype={"concept_id": int})

# Load your condition_occurrence file
cond = pd.read_csv("SCD_Demographics.csv", dtype={"person_id": int, "condition_concept_id": int})

# Filter condition_occurrence to only your complication concept IDs
cond_filt = cond[cond["condition_concept_id"].isin(scd_df["concept_id"])]

# Merge to get complication names
cond_named = cond_filt.merge(
    scd_df,
    left_on="condition_concept_id",
    right_on="concept_id",
    how="left"
)

# Count unique participants per complication
comp_counts = (cond_named.groupby(["Complication", "concept_id"])
               .agg(n_patients=("person_id", "nunique"))
               .reset_index()
               .sort_values("n_patients", ascending=False))

# Get top 10
top10 = comp_counts.head(10)

print(top10)


---
## Step 13 — Assign Severity Scores

**What this does:**  
Assigns each participant a severity category based on their total number
of SCD complications:

| Score | Severity | Criteria |
|-------|----------|----------|
| 0 | None | No complications |
| 1–2 | Mild | 1–2 complications |
| 3–4 | Moderate | 3–4 complications |
| 5+ | Severe | 5 or more complications |

In [ ]:
import pandas as pd

df = pd.read_csv("SCD_Demographics_with_Complications.csv")

# Define target complications (your chosen 10)
target_complications = [
    "Dactylitis",
    "Restrictive lung disease",
    "Preeclampsia",
    "Sickle retinopathy",
    "Osteomyelitis",
    "Aplastic crisis / Aplastic Anemia",
    "Liver cirrhosis",
    "Vitreous haemorrhage",
    "hand and foot syndrome Anemia",
    "Pulmonary hypertension"
]

# Keep only the complications that actually exist in your file
available_cols = [c for c in target_complications if c in df.columns]
missing_cols = [c for c in target_complications if c not in df.columns]

print(" Available complication columns:", available_cols)
print(" Missing complication columns:", missing_cols)

# Step 5: Count how many complications each person has
df["complication_count"] = df[available_cols].sum(axis=1)

# Step 6: Assign severity groups
def classify_severity(count):
    if count == 0:
        return "None"
    elif 1 <= count <= 3:
        return "Mild"
    else:
        return "Severe"

df["severity_group"] = df["complication_count"].apply(classify_severity)

# Select subset (only person_id + complications + severity)
df_severity = df[["person_id"] + available_cols + ["complication_count", "severity_group"]]

# Save
df_severity.to_csv("SCD_Complication_Severity_Only.csv", index=False)

print("✅ Saved SCD_Complication_Severity_Only.csv with shape:", df_severity.shape)
print(df_severity.head())


In [ ]:
import pandas as pd

df = pd.read_csv("SCD_Demographics_with_Complications.cleaned.csv")
print(df.columns.tolist())


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Drop NaNs in severity_group
df_grouped = df_severity.dropna(subset=['severity_group'])

# Compute mean complication prevalence (%) per severity group
severity_group_comps = df_grouped.groupby('severity_group')[complication_cols].mean().T * 100

# Optional: rename complications for readability
complication_labels = {
    "Dactylitis": "Dactylitis",
    "Restrictive_lung_disease": "Restrictive lung disease",
    "Preeclampsia": "Preeclampsia",
    "Sickle_retinopathy": "Sickle retinopathy",
    "Osteomyelitis": "Osteomyelitis",
    "Aplastic_crisis": "Aplastic crisis",
    "Liver_cirrhosis": "Liver cirrhosis",
    "Vitreous_haemorrhage": "Vitreous haemorrhage",
    "Hand_foot_syndrome": "Hand-foot syndrome",
    "Pulmonary_hypertension": "Pulmonary hypertension"
}
severity_group_comps = severity_group_comps.rename(index=complication_labels)

# Plot
severity_group_comps.plot(kind='bar', figsize=(12, 6))
plt.title("SCD Complication Prevalence by Severity Group")
plt.ylabel("Percentage of Patients")
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y')
plt.legend(title='Severity Group')
plt.tight_layout()
plt.show()


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.set(style="whitegrid")

# Bar plot of severity group counts
sns.countplot(data=df_severity, x="severity_group", order=["None", "Mild", "Moderate", "Severe"])
plt.title("SCD Severity Group Distribution")
plt.xlabel("Severity Group")
plt.ylabel("Number of Participants")
plt.show()


In [ ]:
pd.read_csv("SCD_Complication_Severity_Only.csv")["severity_group"].value_counts()


---
## Step 14 — Clean & Finalise Complications Dataset

**What this does:**  
Loads and cleans the full complications dataset, separating demographic
columns from complication columns and ensuring all binary flags are correct (0 or 1).

In [ ]:
import pandas as pd

# Load the original severity file
severity_df = pd.read_csv("SCD_Complication_Severity_Only.csv")

# Check how many rows and the unique values
print(severity_df["severity_group"].value_counts())
print(severity_df.head(476))


In [ ]:
import pandas as pd

# Step 1: Load your demographics + complications file
df = pd.read_csv("SCD_Demographics_with_Complications.cleaned.csv")

# Step 2: Clean column names for consistency
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")

# Step 3: Define top 10 complications (adjust if your column names are different)
top_10_complications = [
    'dactylitis',
    'restrictive_lung_disease',
    'preeclampsia',
    'sickle_retinopathy',
    'osteomyelitis',
    'aplastic_crisis',
    'liver_cirrhosis',
    'vitreous_haemorrhage',
    'hand_foot_syndrome',
    'pulmonary_hypertension'
]

# Step 4: Ensure these complication columns are present
available_cols = [col for col in top_10_complications if col in df.columns]

# Step 5: Count how many complications each person has (i.e., sum of binary flags)
df["complication_count"] = df[available_cols].sum(axis=1)

# Step 6: Assign complication severity group
def classify_severity(count):
    if count == 0:
        return "None"
    elif 1 <= count <= 3:
        return "Mild"
    else:
        return "Moderate"

df["complication_severity_group"] = df["complication_count"].apply(classify_severity)

# Step 7: Save the updated file
df.to_csv("SCD_Complication_Severity_Categorised.csv", index=False)
print("✅ Saved as 'SCD_Complication_Severity_Categorised.csv'")

# Optional: View summary counts
print("\n🧮 Complication Severity Group Counts:")
print(df["complication_severity_group"].value_counts())


In [ ]:
import pandas as pd

df = pd.read_csv("SCD_Demographics_with_Complications.cleaned.csv")

# Drop demographic columns, keep only complication columns (everything after them)
# assuming first 12 columns are demographics (adjust if needed)
complication_cols = df.columns[12:]  

# Step 5: Count how many complications each person has
df["complication_count"] = df[complication_cols].sum(axis=1)

# Step 6: Classify
def classify_severity(count):
    if count == 0:
        return "None"
    elif 1 <= count <= 3:
        return "Mild"
    else:
        return "Moderate"

df["complication_severity_group"] = df["complication_count"].apply(classify_severity)

# Save
df.to_csv("SCD_Complication_Severity_Categorised.csv", index=False)

print("✅ Saved with all complications")
print("\n🧮 Complication Severity Group Counts:")
print(df["complication_severity_group"].value_counts())


In [ ]:
import pandas as pd

# Step 1: Load your dataset
df = pd.read_csv("SCD_Demographics_with_Complications.csv")

# Step 2: Clean column names (optional but recommended)
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")

# Step 3: Define top 10 complications (adjust if your columns use different naming)
top_10_cols = [
    'dactylitis',
    'restrictive_lung_disease',
    'preeclampsia',
    'sickle_retinopathy',
    'osteomyelitis',
    'aplastic_crisis',
    'liver_cirrhosis',
    'vitreous_haemorrhage',
    'hand_foot_syndrome',
    'pulmonary_hypertension'
]

# Step 4: Count how many complications each patient has
df["complication_count"] = df[top_10_cols].sum(axis=1)

# Step 5: Categorize severity based on number of complications
def classify_severity(count):
    if count == 0:
        return "None"
    elif 1 <= count <= 3:
        return "Mild"
    else:
        return "Moderate"

df["complication_severity_group"] = df["complication_count"].apply(classify_severity)

# Step 6: Save or inspect the results
df.to_csv("SCD_Patient_Complication_Severity.csv", index=False)
print("✅ Saved as 'SCD_Patient_Complication_Severity.csv'")
print(df[["person_id", "complication_count", "complication_severity_group"]].head(10))


In [ ]:
import pandas as pd

df = pd.read_csv("SCD_Demographics_with_Complications.csv")

# --- Helper: force complication columns to binary ---
def binarize(d, cols):
    d = d.copy()
    for c in cols:
        d[c] = (d[c].fillna(0) > 0).astype(int)
    return d

# Method 1: Top-10 only (use exact column spellings from your CSV)
top10 = [
    "Restrictive lung disease","Preeclampsia","Sickle retinopathy","Osteomyelitis",
    "Aplastic crisis / Aplastic Anemia","Liver cirrhosis","Vitreous haemorrhage",
    "hand and foot syndrome Anemia","Pulmonary hypertension",
    # include only if it truly exists in your columns:
    # "Dactylitis",
]
top10_avail = [c for c in top10 if c in df.columns]
df1 = binarize(df, top10_avail)
df1["complication_count_top10"] = df1[top10_avail].sum(axis=1)
def grp(x): return "None" if x==0 else ("Mild" if x<=3 else "Moderate")
df1["severity_group_top10"] = df1["complication_count_top10"].apply(grp)

# Method 2: All complication columns = everything after stable demographic set
demographic_cols = [
    'person_id','gender_concept_id','gender','date_of_birth','race_concept_id','race',
    'ethnicity_concept_id','ethnicity','sex_at_birth_concept_id','sex_at_birth',
    'self_reported_category_concept_id','self_reported_category'
]
all_comp_cols = [c for c in df.columns if c not in demographic_cols]
df2 = binarize(df, all_comp_cols)
df2["complication_count_all"] = df2[all_comp_cols].sum(axis=1)
df2["severity_group_all"] = df2["complication_count_all"].apply(grp)

# Summary counts
print("Top-10:", df1["severity_group_top10"].value_counts())
print("All:", df2["severity_group_all"].value_counts())

# Who changed groups?
merged = df1[["person_id","severity_group_top10"]].merge(
    df2[["person_id","severity_group_all"]], on="person_id", how="inner"
)
changed = merged[merged["severity_group_top10"] != merged["severity_group_all"]]
print("\nChanged group (n={}):".format(len(changed)))
print(changed.head(20))


---
## Step 15 — Export Top-10 Complication Details

**What this does:**  
Identifies the top 10 most prevalent SCD complications in the cohort
and exports participant-level detail for these complications.

In [ ]:
demographic_cols = [
    'person_id','gender_concept_id','gender','date_of_birth','race_concept_id','race',
    'ethnicity_concept_id','ethnicity','sex_at_birth_concept_id','sex_at_birth',
    'self_reported_category_concept_id','self_reported_category'
]

top10 = [
    "Restrictive lung disease","Preeclampsia","Sickle retinopathy","Osteomyelitis",
    "Aplastic crisis / Aplastic Anemia","Liver cirrhosis","Vitreous haemorrhage",
    "hand and foot syndrome Anemia","Pulmonary hypertension",
    # "Dactylitis",  # include only if it truly exists in df.columns
]

top10_avail = [c for c in top10 if c in df.columns]
all_comp_cols = [c for c in df.columns if c not in demographic_cols]

print("Top10 used:", top10_avail)
print("All-comp used:", all_comp_cols)
print("Same set? ->", set(top10_avail) == set(all_comp_cols))


In [ ]:
import csv
from pathlib import Path
import pandas as pd
import numpy as np

RAW = Path("SCD_Demographics_with_Complications.csv")
CLEAN = Path("SCD_Demographics_with_Complications.cleaned.csv")

# 1) Read raw lines and keep only the first header; drop any repeated headers or stray lines like '1'
with RAW.open("r", newline="", encoding="utf-8") as f:
    lines = [line.rstrip("\n") for line in f]

if not lines:
    raise ValueError("File is empty.")

header = lines[0]
clean_lines = [header]
for line in lines[1:]:
    # skip exact duplicate header lines
    if line.strip() == header.strip():
        continue
    # skip orphaned lines like just "1" (non-CSV, not the right number of columns)
    if "," not in line:
        continue
    # keep only rows with the same number of columns as header
    if line.count(",") != header.count(","):
        continue
    clean_lines.append(line)

# 2) Write a cleaned temporary CSV
with CLEAN.open("w", newline="", encoding="utf-8") as f:
    for line in clean_lines:
        f.write(line + "\n")

print(f"✅ Wrote cleaned file: {CLEAN}")

# 3) Load with pandas
df = pd.read_csv(CLEAN)

# 4) Basic sanity: drop any residual header-like rows accidentally kept
df = df[df["person_id"].astype(str) != "person_id"]

# 5) Coerce dtypes
# person_id numeric
df["person_id"] = pd.to_numeric(df["person_id"], errors="coerce")
df = df.dropna(subset=["person_id"]).copy()
df["person_id"] = df["person_id"].astype(int)

# complication columns = everything that's not demographic
demographic_cols = [
    'person_id','gender_concept_id','gender','date_of_birth','race_concept_id','race',
    'ethnicity_concept_id','ethnicity','sex_at_birth_concept_id','sex_at_birth',
    'self_reported_category_concept_id','self_reported_category'
]
comp_cols = [c for c in df.columns if c not in demographic_cols]

# 6) Coerce complications to binary 0/1
for c in comp_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0)
    df[c] = (df[c] > 0).astype(int)

# 7) Save back to the original name (and keep a backup)
RAW.rename(RAW.with_suffix(".bak.csv"))
df.to_csv("SCD_Demographics_with_Complications.csv", index=False)

print("Cleaned and saved SCD_Demographics_with_Complications.csv")
print("Complication columns detected:", comp_cols[:10], "..." if len(comp_cols)>10 else "")
print("Shape:", df.shape)


In [ ]:
import pandas as pd

df = pd.read_csv("SCD_Demographics_with_Complications.csv")

demographic_cols = [
    'person_id','gender_concept_id','gender','date_of_birth','race_concept_id','race',
    'ethnicity_concept_id','ethnicity','sex_at_birth_concept_id','sex_at_birth',
    'self_reported_category_concept_id','self_reported_category'
]
comp_cols = [c for c in df.columns if c not in demographic_cols]

# Count + classify
df["complication_count"] = df[comp_cols].sum(axis=1)

def classify_severity(x):
    if x == 0: return "None"
    if 1 <= x <= 3: return "Mild"
    return "Moderate"

df["complication_severity_group"] = df["complication_count"].apply(classify_severity)

# Save a tidy severity-only view (optional)
out = df[["person_id"] + comp_cols + ["complication_count","complication_severity_group"]]
out.to_csv("SCD_Complication_Severity_Only.csv", index=False)

print("✅ Saved SCD_Complication_Severity_Only.csv")
print(df["complication_severity_group"].value_counts())


In [ ]:
import pandas as pd

# Load the dataset
df = pd.read_csv("SCD_Demographics_with_Complications.csv")

# Use the correct capitalized column names
complication_columns = [
    'Dactylitis',
    'Restrictive_lung_disease',
    'Preeclampsia',
    'Sickle_retinopathy',
    'Osteomyelitis',
    'Aplastic_crisis',
    'Liver_cirrhosis',
    'Vitreous_haemorrhage',
    'Hand_foot_syndrome',
    'Pulmonary_hypertension'
]

# Add "Yes"/"No" status columns
for col in complication_columns:
    df[col + '_status'] = df[col].apply(lambda x: "Yes" if pd.notnull(x) and x != 0 else "No")

# Save to new CSV
df.to_csv("SCD_Demographics_with_Complications_Status.csv", index=False)

print("✅ Done! Saved file with Yes/No complication labels.")


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Load your saved data
df = pd.read_csv('SCD_Top10_Complication_Details.csv')

# Count each severity category
severity_counts = df['top10_complication_severity'].value_counts().reindex(['None', 'Mild', 'Severe'], fill_value=0)

# Plot
plt.figure(figsize=(6, 4))
sns.barplot(x=severity_counts.index, y=severity_counts.values, palette='pastel')
plt.title('SCD Complication Severity Distribution (Top 10 Complications)')
plt.xlabel('Severity Category')
plt.ylabel('Number of Participants')
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Load data
df = pd.read_csv('SCD_Top10_Complication_Details.csv')

# Define severity order for consistent bin alignment
severity_order = ['None', 'Mild', 'Severe']
severity_counts = df['top10_complication_severity'].value_counts().reindex(severity_order, fill_value=0)

# Plot histogram (bar plot style for categorical)
plt.figure(figsize=(8, 5))
plt.bar(severity_counts.index, severity_counts.values, color='black', edgecolor='black')

# Title and labels
plt.title('Number of Participants by SCD Complication Severity', fontsize=14)
plt.xlabel('Severity Category', fontsize=12)
plt.ylabel('Number of Participants', fontsize=12)

# Font sizes for ticks
plt.xticks(fontsize=11)
plt.yticks(fontsize=11)

# Annotate counts
for x, y in zip(severity_counts.index, severity_counts.values):
    plt.text(x, y + 5, f'{y}', ha='center', fontsize=10)

# Grid and layout
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd

# Load data
df = pd.read_csv('SCD_Top10_Complication_Details.csv')

# List of complication columns
complication_cols = [
    'dactylitis', 'restrictive_lung_disease', 'preeclampsia',
    'sickle_retinopathy', 'osteomyelitis', 'aplastic_crisis',
    'liver_cirrhosis', 'vitreous_haemorrhage', 'hand_foot_syndrome',
    'pulmonary_hypertension'
]

# Filter only Mild or Severe participants
df_severe_mild = df[df['top10_complication_severity'].isin(['Mild', 'Severe'])].copy()

# Melt for long format: each row = one complication per person
df_long = df_severe_mild.melt(
    id_vars=['person_id', 'top10_complication_severity'],
    value_vars=complication_cols,
    var_name='complication',
    value_name='has_complication'
)

# Keep only complications present (value = 1)
df_present = df_long[df_long['has_complication'] == 1]

# Count complication frequency by severity
complication_summary = df_present.groupby(['top10_complication_severity', 'complication']).size().unstack(fill_value=0)

# Optional: sort columns by total
complication_summary = complication_summary[complication_summary.sum().sort_values(ascending=False).index]

# Display
print(complication_summary)


In [ ]:
import pandas as pd

# Load the data
df = pd.read_csv('SCD_Top10_Complication_Details.csv')

# List of complication columns
complication_cols = [
    'dactylitis', 'restrictive_lung_disease', 'preeclampsia',
    'sickle_retinopathy', 'osteomyelitis', 'aplastic_crisis',
    'liver_cirrhosis', 'vitreous_haemorrhage', 'hand_foot_syndrome',
    'pulmonary_hypertension'
]

# Filter only Mild or Severe
df_mild_severe = df[df['top10_complication_severity'].isin(['Mild', 'Severe'])].copy()

# Create a new column that lists complications for each participant
def get_complication_list(row):
    return [comp for comp in complication_cols if row[comp] == 1]

df_mild_severe['complications_present'] = df_mild_severe.apply(get_complication_list, axis=1)

# Keep only relevant columns
df_result = df_mild_severe[['person_id', 'top10_complication_severity', 'complications_present']]

# Display the result
print(df_result.head(10))


In [ ]:
# Create binary complication matrix for visualization
df_binary = df_mild_severe[['person_id', 'top10_complication_severity'] + complication_cols].copy()
df_binary.set_index(['person_id', 'top10_complication_severity'], inplace=True)

plt.figure(figsize=(12, 6))
sns.heatmap(df_binary, cmap='Greys', cbar=False)
plt.title('Participant-Level Complications (Mild/Severe)')
plt.xlabel('Complication')
plt.ylabel('Participant ID + Severity')
plt.tight_layout()
plt.show()


---
## Step 16 — Severity Heatmap by Complication

**What this does:**  
Creates a detailed heatmap showing complication presence across
severity groups (None/Mild/Moderate/Severe).

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Load your detailed data
df = pd.read_csv('SCD_Top10_Complication_Details.csv')

# List of top 10 complications
complication_cols = [
    'dactylitis', 'restrictive_lung_disease', 'preeclampsia',
    'sickle_retinopathy', 'osteomyelitis', 'aplastic_crisis',
    'liver_cirrhosis', 'vitreous_haemorrhage', 'hand_foot_syndrome',
    'pulmonary_hypertension'
]

# Filter for Mild and Severe participants only
df_filtered = df[df['top10_complication_severity'].isin(['Mild', 'Severe'])]

# Melt to long format
df_long = df_filtered.melt(id_vars='top10_complication_severity',
                           value_vars=complication_cols,
                           var_name='complication',
                           value_name='present')

# Keep only where present == 1
df_present = df_long[df_long['present'] == 1]

# Count by severity and complication
comp_counts = df_present.groupby(['top10_complication_severity', 'complication']).size().reset_index(name='count')

# Plot
plt.figure(figsize=(10, 6))
sns.barplot(data=comp_counts, x='complication', y='count', hue='top10_complication_severity', palette='pastel')
plt.title('Frequency of Each Complication in Mild and Severe Participants')
plt.xlabel('Complication')
plt.ylabel('Number of Participants')
plt.xticks(rotation=45, ha='right', fontsize=10)
plt.legend(title='Severity Group')
plt.tight_layout()
plt.show()


---
## Step 17 — Extend to Full Complication List (Beyond Top 10)

**What this does:**  
Expands the analysis beyond the top 10 to include all SCD complications,
merging with the full reference list.

In [ ]:
import pandas as pd

# Load full SCD complications list (CSV must have 'Complication' and 'concept_id' columns)
scd_comps = pd.read_csv('scd_complications_concept_ids.csv')

# Ensure concept_id is a string
scd_comps['concept_id'] = scd_comps['concept_id'].astype(str)

# Define top 10 complication concept_ids
top10_ids = [
    '137829',   # Aplastic crisis
    '4159748',  # Hand-foot syndrome
    '4322024',  # Pulmonary hypertension
    '4266931',  # Restrictive lung disease
    '4064161',  # Liver cirrhosis
    '141663',   # Osteomyelitis
    '439393',   # Preeclampsia
    '4021365',  # Sickle retinopathy
    '315276',   # Vitreous haemorrhage
    '4131942'   # Dactylitis
]

# Filter out the top 10 complications
non_top10_comps = scd_comps[~scd_comps['concept_id'].isin(top10_ids)].reset_index(drop=True)

# Display result
print(non_top10_comps)

# Optional: Save to CSV
non_top10_comps.to_csv('SCD_NonTop10_Complications.csv', index=False)
print("✅ Saved as 'SCD_NonTop10_Complications.csv'")


In [ ]:
import pandas as pd

# 1. Load full SCD complication list
scd_comps = pd.read_csv('scd_complications_concept_ids.csv')

# 2. Load your participant severity info (from top 10 complications analysis)
df_severity = pd.read_csv('SCD_Top10_Complication_Details.csv')

# 3. Load All of Us condition occurrence file
conditions = pd.read_csv('your_condition_occurrence_data.csv', usecols=['person_id', 'condition_concept_id'])

# Ensure string format for comparison
scd_comps['concept_id'] = scd_comps['concept_id'].astype(str)
conditions['condition_concept_id'] = conditions['condition_concept_id'].astype(str)
df_severity['person_id'] = df_severity['person_id'].astype(str)


In [ ]:
import pandas as pd

# Load your condition occurrence file
cond = pd.read_csv("your_condition_occurrence_data.csv")  # Has person_id and condition_concept_id

# Define complications and their concept IDs
complication_map = {
    "dactylitis": 319835,
    "restrictive_lung_disease": 319843,
    "preeclampsia": 3004295,
    "sickle_retinopathy": 316211,
    "osteomyelitis": 201612,
    "aplastic_crisis": 432961,
    "liver_cirrhosis": 194984,
    "vitreous_haemorrhage": 4226463,
    "hand_foot_syndrome": 320128,
    "pulmonary_hypertension": 316866
}

# Get all unique participants
all_people = cond["person_id"].unique()
result_df = pd.DataFrame(all_people, columns=["person_id"])

# Check each complication
for label, cid in complication_map.items():
    # Flag those with the complication
    affected = cond[cond["condition_concept_id"] == cid]["person_id"].unique()
    result_df[label] = result_df["person_id"].isin(affected).map({True: "Yes", False: "No"})

# Preview
print(result_df.head())


In [ ]:
# List of status columns
status_columns = [col + '_status' for col in complication_columns]

# Count number of 'Yes' values (complications present) per participant
df['complication_count'] = df[status_columns].apply(lambda row: sum(row == 'Yes'), axis=1)


In [ ]:
import pandas as pd

df = pd.read_csv("SCD_Demographics_with_Complications.csv")

# Show all column names
print(df.columns.tolist())


In [ ]:
import pandas as pd

df = pd.read_csv("all_scd_complications_with_counts.csv")
print(df.columns.tolist())


---
## Step 18 — Extract Hydroxyurea (HU) Drug Exposure

**What this does:**  
Queries the All of Us `drug_exposure` table to identify which participants
received Hydroxyurea (HU) treatment — a key SCD disease-modifying drug.

**Output columns added:**
- `HU_exposed` — binary flag (1 = HU exposed, 0 = not exposed)
- Merged into demographics file as `SCD_Demographics_with_HU.csv`

> ⏱️ *BigQuery drug exposure query may take 2–4 minutes.*

In [ ]:
import pandas as pd
from google.cloud import bigquery
import os

client = bigquery.Client()
cdr = os.environ["WORKSPACE_CDR"]

# SQL to pull all Hydroxyurea exposures (not collapsed to flags)
query = f"""
WITH hu_concepts AS (
  SELECT descendant_concept_id AS drug_concept_id
  FROM `{cdr}.concept_ancestor`
  WHERE ancestor_concept_id = 1377141
  UNION DISTINCT SELECT 1377141
)
SELECT
  de.person_id,
  de.drug_exposure_start_date,
  de.drug_exposure_end_date,
  de.drug_concept_id
FROM `{cdr}.drug_exposure` de
JOIN hu_concepts hu
  ON de.drug_concept_id = hu.drug_concept_id
ORDER BY de.person_id, de.drug_exposure_start_date
"""

hu_exposures = client.query(query).to_dataframe()
print(" Hydroxyurea exposures pulled:", hu_exposures.shape)

# How many HU exposures per person
hu_counts = (
    hu_exposures.groupby("person_id")
                .size()
                .reset_index(name="num_hu_exposures")
)

# Merge counts back to exposures for convenience
hu_exposures = hu_exposures.merge(hu_counts, on="person_id", how="left")

# Save both detailed exposures and counts
hu_exposures.to_csv("Hydroxyurea_exposures.csv", index=False)
hu_counts.to_csv("Hydroxyurea_counts.csv", index=False)

print(" Saved Hydroxyurea_exposures.csv (all start/end per person)")
print(" Saved Hydroxyurea_counts.csv (number of HU records per person)")

# Quick look
print(hu_counts.head())


In [ ]:
import pandas as pd
from google.cloud import bigquery
import os

client = bigquery.Client()
cdr = os.environ["WORKSPACE_CDR"]

# --- SQL: all unique Hydroxyurea patients ---
query = f"""
WITH hu_concepts AS (
  SELECT descendant_concept_id AS drug_concept_id
  FROM `{cdr}.concept_ancestor`
  WHERE ancestor_concept_id = 1377141
  UNION DISTINCT SELECT 1377141
)
SELECT DISTINCT de.person_id
FROM `{cdr}.drug_exposure` de
JOIN hu_concepts hu
  ON de.drug_concept_id = hu.drug_concept_id
"""

hu_people = client.query(query).to_dataframe()
print(" Found Hydroxyurea patients:", hu_people.shape[0])

# --- Load demographics ---
demog = pd.read_csv("SCD_Demographics.csv")

# --- Merge HU flag directly ---
demog["on_hydroxyurea"] = demog["person_id"].isin(hu_people["person_id"]).map({True: "Yes", False: "No"})

# --- Save ---
demog.to_csv("SCD_Demographics_with_HU.csv", index=False)

# Quick summary
print(demog["on_hydroxyurea"].value_counts())


In [ ]:
import pandas as pd

# Load the file you just saved
check = pd.read_csv("SCD_Demographics_with_HU.csv")

print("Shape:", check.shape)
print(check.head())

# Check how many are HU exposed vs not
if "HU_exposed" in check.columns:   # or whatever column name you added
    print(check["HU_exposed"].value_counts())


In [ ]:
from google.cloud import bigquery
import os

client = bigquery.Client()
cdr = os.environ["WORKSPACE_CDR"]

concept_id = 19007385

query = f"""
SELECT concept_id, concept_name, concept_class_id, vocabulary_id
FROM `{cdr}.concept`
WHERE concept_id = {concept_id}
"""

df = client.query(query).to_dataframe()
print(df)


In [ ]:
from google.cloud import bigquery
import pandas as pd, os

client = bigquery.Client()
cdr = os.environ["WORKSPACE_CDR"]

query = f"""
WITH hu_concepts AS (
  SELECT descendant_concept_id AS drug_concept_id
  FROM `{cdr}.concept_ancestor`
  WHERE ancestor_concept_id = 1377141
  UNION DISTINCT SELECT 1377141
),
hu AS (
  SELECT
    de.person_id,
    MIN(CAST(de.drug_exposure_start_date AS DATE)) AS hu_first_date
  FROM `{cdr}.drug_exposure` de
  JOIN hu_concepts hu
    ON de.drug_concept_id = hu.drug_concept_id
  GROUP BY de.person_id
)
SELECT * FROM hu
"""
hu_people = client.query(query).to_dataframe()
hu_people["on_hydroxyurea"] = "Yes"


In [ ]:
import pandas as pd
from google.cloud import bigquery
import os

# Load your cohort
demog = pd.read_csv("SCD_Demographics.csv")
cohort_ids = demog["person_id"].dropna().astype(int).tolist()

client = bigquery.Client()
cdr = os.environ["WORKSPACE_CDR"]

# Build an IN list for your cohort (safe with a few hundred IDs)
id_list = ",".join(map(str, cohort_ids))

# Ingredient-only (no descendants): concept_id = 1377141
query_ids = f"""
SELECT DISTINCT de.person_id
FROM `{cdr}.drug_exposure` de
WHERE de.drug_concept_id = 1377141
  AND de.person_id IN ({id_list})
"""

hu_people_ingredient_only = client.query(query_ids).to_dataframe()

# Merge a Yes/No flag into your cohort (ingredient-only)
demog_ing = demog.copy()
demog_ing["on_hydroxyurea_ingredient_only"] = demog_ing["person_id"].isin(
    hu_people_ingredient_only["person_id"]
).map({True: "Yes", False: "No"})

print(demog_ing["on_hydroxyurea_ingredient_only"].value_counts())
demog_ing.to_csv("SCD_Demographics_HU_ingredient_only.csv", index=False)


In [ ]:
import pandas as pd
from google.cloud import bigquery
import os, math

client = bigquery.Client()
cdr = os.environ["WORKSPACE_CDR"]

# 1) Load your cohort person_ids (from your CSV)
cohort = pd.read_csv("SCD_Demographics.csv", usecols=["person_id"])
person_ids = cohort["person_id"].dropna().astype(int).unique().tolist()

# 2) Helper to run the query in chunks (BigQuery has a 1MB query text limit)
def fetch_hu_first_last(ids, gap_days=30):
    results = []
    CHUNK = 5000  # 476 fits in one chunk; keep general.
    for i in range(0, len(ids), CHUNK):
        chunk = ids[i:i+CHUNK]
        query = f"""
        -- Hydroxyurea concept set
        WITH hu_concepts AS (
          SELECT descendant_concept_id AS drug_concept_id
          FROM `{cdr}.concept_ancestor`
          WHERE ancestor_concept_id = 1377141
          UNION DISTINCT SELECT 1377141
        )
        SELECT
          de.person_id,
          MIN(CAST(de.drug_exposure_start_date AS DATE)) AS hu_first_start_date,
          MAX(CAST(COALESCE(de.drug_exposure_end_date, de.drug_exposure_start_date) AS DATE)) AS hu_last_end_date,
          COUNT(*) AS hu_records
        FROM `{cdr}.drug_exposure` de
        JOIN hu_concepts hu
          ON de.drug_concept_id = hu.drug_concept_id
        WHERE de.person_id IN UNNEST(@pid_list)
        GROUP BY de.person_id
        """
        job = client.query(
            query,
            job_config=bigquery.QueryJobConfig(
                query_parameters=[bigquery.ArrayQueryParameter("pid_list", "INT64", chunk)]
            ),
        )
        results.append(job.to_dataframe())
    return pd.concat(results, ignore_index=True) if results else pd.DataFrame()

hu_first_last = fetch_hu_first_last(person_ids)
print("Rows returned:", len(hu_first_last))
hu_first_last.head(10)


---
## ✅ Notebook 02 Complete

**Output files created:**
- `SCD_Demographics_with_Complications.csv` — demographics + binary complication flags
- `SCD_Complication_Severity_Only.csv` — severity scores per participant
- `SCD_Demographics_with_HU.csv` — HU drug exposure flags
- `scd_complication_counts_all.csv` — complication prevalence counts
- `SCD_Top10_Complication_Details.csv` — top 10 complication detail

**Next step:** Open and run `03_lab_genotypic_extraction.ipynb`

---
> 💡 **Tip:** If a cell fails with `NameError: demo_with_complications`, 
> run all cells from the top using **Kernel → Restart & Run All**.